# 14 — The real thesis run

**In one sentence:** all 5 ways of answering on the fixed **234 test problems**, plus a better LoRA,
inside the Colab budget — the numbers that go into the thesis.

~~~text
234 test problems = HumanEval+ 164 (easy) + LiveCodeBench 70 (31 easy + 39 medium)
ways: thinking OFF · ON · "think briefly" · thinking limit (1,024) · LoRA-1 (mini) · LoRA-2 (new)
~~~

## How to run it (read once)

| | |
|---|---|
| **Where** | **Everything runs on the A100.** No T4 needed: a 2-minute smoke test on the A100 replaces it. |
| **Start** | Runtime → Change runtime type → **A100 GPU**. Then **Runtime → Run all**. |
| **You type** | Only once: the units left (Colab shows it under **RAM/Disk → View resources**). |
| **You never edit** | Every number is in the **config cell** (step 2). The notebook decides RUN/SKIP itself. |
| **If it disconnects** | **Run all** again. Every stage skips what is already finished. |
| **When it ends** | **Runtime → Disconnect and delete runtime.** An idle A100 still costs 5.3 units/hour. |
| **Paste back** | Only the output of the **last cell** (step 12). |

## The fixed rules (DECISIONS #65, #66 — written before any result)

| Rule | Value | Why |
|---|---|---|
| Token limit, HumanEval+ | **4,096** | easy; the mini-thesis cut-offs at 4,096 were mostly loops |
| Token limit, LiveCodeBench | **8,192** | medium problems need longer honest reasoning |
| Same for every way | yes | fairness (`CLAUDE.md` §4) |
| Thinking limit (the "limit" way) | **1,024** thinking tokens | about the LoRA's thinking length in the mini-thesis, so it tests "is a hard cut as good as training?" |
| Main LoRA | **LoRA-2** if it gets trained, else LoRA-1 | fixed now, so we can't pick the best-looking one later |
| Grading | the benchmarks' own **plus** tests, in a separate process | never by eye (`CLAUDE.md` §4) |
| Budget | start a stage only if *units left − its cost ≥ 20* | never run out (DECISIONS #65) |

**Where we are:** `PROBLEM ✅ → GAP ✅ → QUESTION ✅ → HYPOTHESIS ✅ → EXPERIMENT ⬅ HERE → RESULTS`

## 1. Install
**Problem:** Colab starts empty. **Why:** we need Unsloth (training), evalplus, datasets, and
flash-linear-attention (half of Qwen3.5's **fast path**). **In:** nothing. **Out:** installed packages.
**Why this way:** Unsloth goes first because it chooses its own library versions; every later
step then uses the same versions. Nothing imports torch before this cell, so no restart is needed.

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q evalplus datasets flash-linear-attention

## 2. Config, GPU check, Drive, code, budget
**Problem:** every number must live in one place. **Why:** so nothing is edited mid-run.
**In:** the units left (typed once). **Out:** paths, limits, batch size, the budget clock.
**Why this way:** the batch size comes from the GPU's memory; if a batch still doesn't fit, the
answering script splits it in two by itself (it slows down, it doesn't stop). Any failed command
stops the notebook with a clear message instead of carrying on with half the data.

In [ ]:
import os, sys, json, glob, shutil, subprocess
from IPython import get_ipython

name, mem = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout.strip().split(", ")
GPU, GPU_GB = name, float(mem) / 1024
PRODUCTION_OK = ("A100" in GPU or "H100" in GPU) and GPU_GB > 35
BATCH = 128 if GPU_GB > 35 else 16
print(f"GPU: {GPU} ({GPU_GB:.0f} GB) · batch {BATCH} · "
      + ("OK for the real run" if PRODUCTION_OK else "NOT an A100: only the smoke test will run"))

def rows(path):
    return sum(1 for l in open(path) if l.strip()) if os.path.exists(path) else 0

def sh(cmd):
    """Run a shell command; STOP the notebook if it fails."""
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"FAILED (read the lines above): {cmd[:120]}")

from google.colab import drive
drive.mount("/content/drive")
REPO = "https://github.com/mahmudulhaquequdrati/stop-overthinking-thesis.git"
if not os.path.isdir("/content/thesis"):
    sh(f"git clone -q {REPO} /content/thesis")
else:
    sh("cd /content/thesis && git pull -q")
os.chdir("/content/thesis"); sys.path.insert(0, "/content/thesis/scripts")

D      = "/content/drive/MyDrive/stop-overthinking/results"
T      = f"{D}/thesis"                        # everything from this notebook
LORA1  = f"{D}/mini/lora/lora100"             # the mini-thesis LoRA (65% on MBPP+)
LORA2  = f"{T}/lora/lora2"                    # the new LoRA (stage B)
WHEELS = "/content/drive/MyDrive/stop-overthinking/wheels"
os.makedirs(T, exist_ok=True); os.makedirs(WHEELS, exist_ok=True)
assert os.path.exists(f"{LORA1}/adapter_config.json"), f"LoRA-1 not found at {LORA1} - STOP"

MAXTOK       = {"he": 4096, "lcb": 8192}     # the same for every way (DECISIONS #66)
THINK_BUDGET = 1024                           # the "limit" way
SRC          = {"he": "humaneval", "lcb": "lcb"}
WAYS = {"off": ("thinking_off", None), "on": ("thinking_on", None), "brief": ("brief", None),
        "limit": ("limit", None), "lora1": ("thinking_on", LORA1), "lora2": ("thinking_on", LORA2)}
COST = {"smoke": 0.7, "A": 10.3, "B1": 7.2, "B2": 1.0, "B3": 2.5, "C": 12.8, "D": 2.7}   # units, DECISIONS #66

import budget
budget.start(float(input("Units left right now (Colab -> RAM/Disk -> View resources): ")),
             f"{T}/budget-ledger.jsonl")

## 3. The fast path: causal-conv1d (built once, then kept on Drive)
**Problem:** without `causal-conv1d` Qwen3.5 runs slow backup code. **Why:** speed = units.
**In:** nothing. **Out:** "FAST PATH ON". **Why this way:** building it takes ~10 minutes, so we
build it **once** and keep the result on Drive; later sessions install it in seconds. If the kept
copy doesn't fit a newer Colab, it is rebuilt automatically.

In [ ]:
def fast_path_ok():
    return subprocess.run([sys.executable, "-c", "import causal_conv1d, fla"],
                          capture_output=True).returncode == 0

for attempt in (1, 2):
    if fast_path_ok():
        break
    whl = glob.glob(f"{WHEELS}/causal_conv1d-*.whl")
    if not whl:
        print("building causal-conv1d once (~10 min), then kept on Drive ...")
        get_ipython().system(f"pip wheel -q causal-conv1d --no-build-isolation --no-deps -w {WHEELS}")
        whl = glob.glob(f"{WHEELS}/causal_conv1d-*.whl")
    if whl:
        get_ipython().system(f"pip install -q {whl[0]}")
    if not fast_path_ok():
        for w in whl: os.remove(w)          # made for an older Colab: build again
print("FAST PATH ON" if fast_path_ok() else "fast path OFF - it still runs, only slower")

## 4. Problems, the overlap check, the grader check
**Problem:** we need the test set, the training pools, a proof that they don't overlap, and a
grader we can trust. **Why:** a leaked test problem or a broken grader ruins every number.
**In:** HumanEval+, LiveCodeBench, MBPP+ from Hugging Face. **Out:** 4 data files, the list of
excluded training problems, and "grader OK".
**Why this way:** the grader must pass the benchmarks' **official solutions** first. The
overlap check is strict: leaving out a few training problems costs nothing.

In [ ]:
for c in ["python scripts/test_prompts.py", "python scripts/build_problem_set.py",
          "python scripts/mbpp_data.py", "python scripts/lcb_train_data.py", "python scripts/overlap_check.py",
          "python scripts/grade_plus.py --check-official 30 --problems data/problems.json --source humaneval",
          "python scripts/grade_plus.py --check-official 20 --problems data/mbpp.json"]:
    sh(c)
N = {ds: sum(p["source"] == SRC[ds] for p in json.load(open("data/problems.json"))["problems"]) for ds in SRC}
print("test problems:", N, "(must be he 164, lcb 70)")
assert N == {"he": 164, "lcb": 70}, "the test set is not the fixed 234 problems - STOP"

## 5. Helpers (answer, grade in the background, count what is done)
**Problem:** the same few steps run many times. **Why:** one tested way to do them, no copies.
**In/Out:** see each function. **Why this way:**
- `answer()` skips a file that is already complete, without even loading the model.
- `grade()` runs **in the background** on the CPU while the GPU answers the next way — so grading
  costs no extra GPU time. `wait_grading()` waits for all of it.
- Any failure (e.g. a LoRA that loads empty) **stops the notebook** with a clear message.

In [ ]:
BG = []

def tail(ans):
    lines = [l for l in open(ans + ".grade.log").read().split("\n") if l.strip() and not l.startswith("saved")]
    return lines[-1] if lines else "(no grading output)"

def ans_path(way, ds, folder=None, prefix="test"):
    return f"{folder or T}/{prefix}-{way}-{ds}.jsonl"

def answer(way, ds, tries, folder=None, n=0, maxtok=None, budget_tok=None, only_ids=None, prefix="test"):
    """Generate answers for one way on one dataset. Skips if already complete."""
    out = ans_path(way, ds, folder, prefix)
    want = (len(json.load(open(only_ids))) if only_ids else (n or N[ds])) * tries
    if rows(out) >= want:
        print(f"  {way:<6}{ds:<4} complete ({rows(out)} answers) - skipped"); return out
    policy, adapter = WAYS[way] if way in WAYS else ("thinking_on", None)
    cmd = (f'python scripts/gen_colab.py --policy {policy} --label {way} --problems data/problems.json '
           f'--source {SRC[ds]} --samples {tries} --dtype auto --batch {BATCH} '
           f'--max-tokens {maxtok or MAXTOK[ds]} --think-budget {budget_tok or THINK_BUDGET} --out "{out}"')
    if n: cmd += f" --n {n}"
    if adapter: cmd += f' --adapter "{adapter}"'
    if only_ids: cmd += f' --only-ids "{only_ids}"'
    print(f"  {way:<6}{ds:<4} answering ...", flush=True)
    sh(cmd)
    return out

def graded_ok(ans):
    g = ans.replace(".jsonl", "-graded.csv")
    return os.path.exists(g) and rows(g) - 1 == rows(ans) and os.path.getmtime(g) >= os.path.getmtime(ans)

def grade(ans, ds, problems="data/problems.json", wait=False):
    for a, p in BG:                       # never two graders on the same file
        if a == ans: p.wait()
    if graded_ok(ans): return
    script = "grade_plus.py" if ds in ("he", "mbpp") else "grade_lcb.py"
    p = subprocess.Popen([sys.executable, f"scripts/{script}", "--answers", ans, "--problems", problems],
                         stdout=open(ans + ".grade.log", "w"), stderr=subprocess.STDOUT)
    if wait: p.wait(); print("  " + tail(ans))
    else: BG.append((ans, p))

def wait_grading():
    for ans, p in BG:
        p.wait()
        print("  " + tail(ans))
    BG.clear()

def stage(name, ways, tries):
    """Run a test stage: every way on both datasets, grading in the background."""
    todo = [(w, ds) for w in ways for ds in SRC if rows(ans_path(w, ds)) < N[ds] * tries]
    if not todo:
        print(f"stage {name}: already complete"); return
    assert PRODUCTION_OK, "Production stages need the A100 (step 2 said NOT an A100)."
    if not budget.gate(f"stage {name}", COST[name]): return
    for w, ds in todo:
        grade(answer(w, ds, tries), ds)
    budget.done(f"stage {name}")
print("helpers ready")

## 6. ⚠️ SAFETY NET — the smoke test (A100, ~5 minutes, ~0.7 units)
**Problem:** a long run that is broken from the first second wastes units. **Why:** this replaces
the T4 test: it runs **every way** (incl. both LoRA loads and the limit way) on **2 problems per
dataset**, with a tiny token limit, and grades them. Accuracy here means nothing; **working** does.
**In:** 2 HumanEval+ + 2 LiveCodeBench problems. **Out:** "SMOKE PASSED" or a clear stop.
**Why this way:** every code path the real run uses, for the price of a few minutes.

In [ ]:
SM = f"{T}/smoke"
if os.path.exists(f"{SM}/PASSED"):
    print("smoke test already passed in an earlier session - skipped")
elif budget.gate("smoke test", COST["smoke"]):
    shutil.rmtree(SM, ignore_errors=True); os.makedirs(SM)
    smoke_ways = ["off", "on", "brief", "limit", "lora1"] + (["lora2"] if os.path.exists(f"{LORA2}/adapter_config.json") else [])
    for w in smoke_ways:
        for ds in SRC:
            grade(answer(w, ds, 1, folder=SM, n=2, maxtok=256, budget_tok=128), ds, wait=True)
    for w in smoke_ways:
        for ds in SRC:
            r = [json.loads(l) for l in open(ans_path(w, ds, SM))]
            assert len(r) == 2, f"{w} {ds}: expected 2 answers"
            if w == "off": assert all(x["thinking_tokens"] == 0 for x in r), "OFF has thinking - STOP"
            else:          assert any(x["thinking_tokens"] > 0 for x in r), f"{w} counted 0 thinking - STOP"
            assert graded_ok(ans_path(w, ds, SM)), f"{w} {ds} was not graded - STOP"
    open(f"{SM}/PASSED", "w").write("ok")
    budget.done("smoke test")
    print("\nSMOKE PASSED - every way answers, counts thinking and gets graded")

## 7. Stage A — the 5 ways on the 234 test problems, try 1 (the must-have)
**Problem:** the core result. **Why:** even if everything after this is skipped, this alone
answers the research question (with LoRA-1).
**In:** 234 problems × OFF, ON, brief, limit, LoRA-1, 1 try each. **Out:** 10 answer files, graded.
**Why this way:** grading runs in the background while the next way is being answered.
Estimated ~10 units (a batch waits for its slowest answer, capped by the token limit).

In [ ]:
stage("A", ["off", "on", "brief", "limit", "lora1"], tries=1)

## 8. Stage B — a better LoRA (LoRA-2): more data, and medium-style problems
**Problem:** LoRA-1 learned only from ~100 easy MBPP+ problems; our test also has **medium**
problems. **Why:** the thesis needs improvement where it is hardest.
**In:** 200 MBPP+ problems + 80 older LiveCodeBench problems (40 easy + 40 medium, all from before
2025, so never test problems), 4 tries each. **Out:** `train-set-v2.jsonl` → LoRA-2 → tested.
**Why this way:** the same shortest-correct method as the mini-thesis, same settings, 3 epochs.
The mini-thesis answers are reused (no paying twice). Problems flagged by the overlap check are
left out. B1 ≈ 7 units, B2 ≈ 1, B3 ≈ 2.5.

In [ ]:
TR_MBPP, TR_LCB = f"{T}/train-mbpp.jsonl", f"{T}/train-lcb.jsonl"
TRAIN_V2 = f"{T}/train-set-v2.jsonl"

# B1: training answers (4 tries each)
if os.path.exists(TRAIN_V2):
    print("B1: training set already made - skipped")
else:
    assert PRODUCTION_OK, "Stage B needs the A100."
    if budget.gate("B1 training answers", COST["B1"]):
        if not os.path.exists(TR_MBPP) and os.path.exists(f"{D}/mini/train-answers.jsonl"):
            shutil.copy(f"{D}/mini/train-answers.jsonl", TR_MBPP)        # reuse the mini-thesis answers
        for split in ("train", "test"):
            sh(f'python scripts/gen_colab.py --policy thinking_on --label train --problems data/mbpp.json '
               f'--source mbpp --split {split} --samples 4 --dtype auto --batch {BATCH} --max-tokens 4096 --out "{TR_MBPP}"')
        sh(f'python scripts/gen_colab.py --policy thinking_on --label train --problems data/lcb_train.json '
           f'--source lcb --samples 4 --dtype auto --batch {BATCH} --max-tokens 8192 --out "{TR_LCB}"')
        grade(TR_MBPP, "mbpp", problems="data/mbpp.json", wait=True)
        grade(TR_LCB, "lcb", problems="data/lcb_train.json", wait=True)
        sh(f'python scripts/make_train_set.py --answers "{TR_MBPP}" --problems data/mbpp.json '
           f'--splits train,test --exclude data/overlap_exclude.json --out "{T}/train-set-mbpp.jsonl"')
        sh(f'python scripts/make_train_set.py --answers "{TR_LCB}" --problems data/lcb_train.json '
           f'--exclude data/overlap_exclude.json --out "{T}/train-set-lcb.jsonl"')
        with open(TRAIN_V2, "w") as f:
            for part in ("mbpp", "lcb"):
                f.write(open(f"{T}/train-set-{part}.jsonl").read())
        print(f"train-set-v2: {rows(TRAIN_V2)} examples")
        budget.done("B1 training answers")

# B2: train LoRA-2
if os.path.exists(f"{LORA2}/adapter_config.json"):
    print("B2: LoRA-2 already trained - skipped")
elif os.path.exists(TRAIN_V2) and budget.gate("B2 train LoRA-2", COST["B2"]):
    sh(f'python scripts/train_lora.py --train "{TRAIN_V2}" --fraction 1.0 --epochs 3 --out "{LORA2}"')
    budget.done("B2 train LoRA-2")

# B3: test LoRA-2, try 1
if os.path.exists(f"{LORA2}/adapter_config.json"):
    stage("B3", ["lora2"], tries=1)

## 9. Stage C — a second try for every way (tighter error bars)
**Problem:** 1 try per problem is noisy. **Why:** 2 tries make the error bars narrower.
**In:** the same 234 problems, try 2, all 6 ways. **Out:** the files grow to 2 tries each.
**Why this way:** the script adds only the missing try (try 1 is kept). ~12.8 units; skipped
automatically if it would break the 20-unit floor.

In [ ]:
ways_c = ["off", "on", "brief", "limit", "lora1"] + (["lora2"] if os.path.exists(f"{LORA2}/adapter_config.json") else [])
stage("C", ways_c, tries=2)

## 10. Stage D — would thinking ON have finished with 16,384 tokens?
**Problem:** an examiner will ask: "Did your token limit unfairly cut off thinking ON?"
**Why:** we answer with a measurement, not a guess.
**In:** only the thinking-ON answers of try 1 that were cut off. **Out:** how many of them finish,
and how many are then correct, with 16,384 tokens.
**Why this way:** re-running only the cut-off answers costs ~2.7 units; raising the limit for
everything would cost 4× the whole run.

In [ ]:
import csv
wait_grading()
done_d = all(os.path.exists(ans_path("on16k", ds, prefix="x16k")) for ds in SRC)
if not done_d and PRODUCTION_OK and budget.gate("D: 16k re-run", COST["D"]):
    for ds in SRC:
        g = ans_path("on", ds).replace(".jsonl", "-graded.csv")
        cut_ids = sorted({r["task_id"] for r in csv.DictReader(open(g))
                          if r["hit_limit"] == "True" and r["sample_index"] == "0"})
        ids_file = f"{T}/x16k-ids-{ds}.json"; json.dump(cut_ids, open(ids_file, "w"))
        print(f"  {ds}: {len(cut_ids)} thinking-ON answers were cut off at {MAXTOK[ds]}")
        if cut_ids:
            grade(answer("on16k", ds, 1, maxtok=16384, only_ids=ids_file, prefix="x16k"), ds, wait=True)
        else:
            open(ans_path("on16k", ds, prefix="x16k"), "w").close()
    budget.done("D: 16k re-run")
for ds in SRC:
    p = ans_path("on16k", ds, prefix="x16k")
    if rows(p):
        g = list(csv.DictReader(open(p.replace(".jsonl", "-graded.csv"))))
        fin = sum(r["hit_limit"] == "False" for r in g); ok = sum(r["passed"] == "True" for r in g)
        print(f"  {ds}: of {len(g)} cut-off ON answers, {fin} finished within 16,384 and {ok} were correct "
              f"(= +{100 * ok / N[ds]:.1f} points for ON on {ds} with a 16k limit)")

## 11. Finish grading
**Problem:** background grading may still be running, or was lost in a disconnect.
**Why:** the results need every file graded. **In:** all answer files. **Out:** all graded.
**Why this way:** anything not graded (or graded before its file grew) is graded now.
This needs only the CPU — if the budget is tight, you may switch to a CPU runtime first
(then run steps 1, 2 (answer the units question), 4 and 5, then this cell).

In [ ]:
wait_grading()
for way in WAYS:
    for ds in SRC:
        a = ans_path(way, ds)
        if rows(a) and not graded_ok(a):
            grade(a, ds, wait=True)
print("all graded")

## 12. The result — paste this output into the chat
**Problem:** turn ~2,000 graded answers into the thesis answer. **Why:** this is the results chapter.
**In:** every graded file. **Out:** tables for all 234 problems, HumanEval+, LCB easy, LCB medium;
hypotheses H1–H3 for the main LoRA; `summary.csv` on Drive; the budget ledger.
**Why this way:** paired error bars, the main LoRA fixed in advance, rules from PLAN §5.

👉 **Then: Runtime → Disconnect and delete runtime.**

In [ ]:
!python scripts/compare_thesis.py --dir "{T}"
print("\nbudget ledger (estimates; Colab's own number is the truth):")
for l in open(f"{T}/budget-ledger.jsonl").read().strip().split("\n")[-8:]:
    print("  ", l)
print(f"\n~{budget.left():.1f} units left (estimate). NOW: Runtime -> Disconnect and delete runtime.")